In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN attached.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- bisel8 on THPep and CellPPD, 3 seeds each. ~1 h, one notebook.
#
# Finishes the classification side of the sweep. Both use their LoRA script and
# both lack a val file, so each job takes the 5-fold CV branch: five training runs
# predicting the full test set, ensembled by mean logit.
#
# COST (8 blocks, batch 32, measured or scaled from measured):
#     THPep   ~10 min per seed   (487 train, 88 tokens, 5 folds)
#     CellPPD ~26 min per seed   (1164 train, 95 tokens, 5 folds)
#     6 jobs, two GPUs -> about an hour.
#
# READ THE CellPPD RESULT WITH CARE. Bag-of-tokens -- counting the 405 vocabulary
# items, no transformer at all -- scores 0.8270 there against the full 337M
# encoder's 0.8278. CellPPD cannot distinguish two backbones, so whatever bisel8
# scores is not evidence about its representation. It is included for completeness
# against the paper's benchmark list, not as a measurement. THPep clears its own
# control (0.6854) comfortably and is the informative one of the pair, though its
# 122-molecule test set gives a bootstrap CI near +-0.15.
subprocess.run('pip install -q -U "transformers>=5.0" lightning peft', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time, threading
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
assert torch.cuda.device_count() >= 2, "this plan pins jobs to cuda:0 and cuda:1"


In [ ]:

# -- Cell 3 -- their code and data, our code, both released models.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local), shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)
print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

TRAIN_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"
DATA_DIR = REPO + "/data"
if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH), ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)

# bench_control.py writes THPep_train.csv / THPep_test.csv, which their repo does
# not ship and their script needs. Deterministic (stratified 80/20, seed 101), so
# it reproduces the same split every run and across accounts.
r = subprocess.run(["python", "bench_control.py"], cwd=CODE, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-1500:]
for p in (TRAIN_PY, DATA_DIR + "/THPep_train.csv", DATA_DIR + "/CellPPD_train.csv"):
    assert os.path.exists(p), "missing: " + p
print("ok -- both datasets ready")


In [ ]:

# -- Cell 4 -- bisel8, pulled from Drive so every run uses identical weights.
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)
ARM = EXPORT + "/peptideclm-2-mlm-bisel8"

if not os.path.exists(ARM + "/model.safetensors"):
    os.makedirs(ARM, exist_ok=True)
    subprocess.run("rclone copy %s/models/peptideclm-2-mlm-bisel8 %s -P" % (REMOTE, ARM),
                   shell=True, check=False)
if os.path.exists(ARM + "/model.safetensors"):
    print("bisel8 <- Drive")
else:
    print("not on Drive, deriving from the teacher")
    r = subprocess.run(["python", "export_truncated.py", "--out", ARM,
                        "--keep", "0,1,2,3,5,6,10,16"], cwd=CODE,
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-1200:]

KEEP_LIST = [0, 1, 2, 3, 5, 6, 10, 16]
cfg = json.load(open(ARM + "/config.json"))
assert cfg.get("pruned_from", {}).get("kept") == KEEP_LIST, \
    "wrong blocks in config: %s" % cfg.get("pruned_from")
# VERIFY THE WEIGHTS, NOT JUST THE CONFIG. A correct config.json sitting next to
# the wrong model.safetensors already cost this project four mislabelled
# benchmark runs. verify_weights compares exported block j against TEACHER block
# keep[j], tensor by tensor, so a swapped checkpoint cannot survive it.
_v = subprocess.run(["python", "-c",
                     "import sys; sys.path.insert(0, '.');"
                     "from export_truncated import verify_weights;"
                     "verify_weights(%r, %r, %r)" % (TEACH, ARM, KEEP_LIST)],
                    cwd=CODE, capture_output=True, text=True)
print(_v.stdout.strip() or _v.stderr[-800:])
assert _v.returncode == 0, (
    "THESE ARE NOT THE bisel8 WEIGHTS -- re-upload "
    "models/peptideclm-2-mlm-bisel8 to Drive.")

print("bisel8: %d blocks, %.1f MB" % (cfg["num_blocks"],
      os.path.getsize(ARM + "/model.safetensors") / 1e6))


In [ ]:

# -- Cell 5 -- the plan: one queue per GPU, balanced by expected cost.
#
# CellPPD jobs are ~2.6x a THPep job, so the queues are split to finish together
# rather than by benchmark. Pinned rather than work-stealing, matching the sweep
# notebooks.
SEEDS = [101, 202, 303]
# RERUN: THPep SEED 101 ONLY -- the single job still missing. CellPPD 101/202/303
# and THPep 202/303 all produced results in the previous run.
#
# It was the LAST job in gpu0's queue last time, which is worth keeping in mind:
# a session ending before the queue drained looks identical to a job that failed.
# If it fails again the 40-line log below will say which.
#
# Cell 5 pulls from Drive first and skips any job whose results already exist, so
# nothing that finished can be clobbered by this run.
PLAN = {0: [("THPep", 101)]}       # ~10 min, gpu1 unused
# Full plan, for reference:
#   PLAN = {0: [("CellPPD", 101), ("CellPPD", 202), ("THPep", 101)],
#           1: [("CellPPD", 303), ("THPep", 202), ("THPep", 303)]}
OUT = WORK + "/results/bisel8_cls"
os.makedirs(OUT, exist_ok=True)
BS = "32"

subprocess.run("rclone copy %s/results/bisel8_cls %s --transfers 8 -P" % (REMOTE, OUT),
               shell=True, check=False)

def job_dir(bench, seed):
    return "%s/%s/seed_%d" % (OUT, bench, seed)

def job_done(bench, seed):
    return bool(glob.glob(job_dir(bench, seed) + "/*_results.csv"))

for gpu, jobs in PLAN.items():
    pend = [j for j in jobs if not job_done(*j)]
    print("gpu%d: %s  (%d pending)" % (gpu, ["%s/%d" % j for j in jobs], len(pend)))


In [ ]:

# -- Cell 6 -- run.
#
# --gpu_index must be passed: their Trainer does devices=[int(args.gpu_index)] on
# the raw argument, whose default is None.
t0 = time.time()
state = {g: "starting" for g in PLAN}

def run_queue(gpu, jobs):
    for bench, seed in jobs:
        tag = "%s s%d" % (bench, seed)
        if job_done(bench, seed):
            print("[%5.1f min] gpu%d skip %s" % ((time.time()-t0)/60, gpu, tag)); continue
        d = job_dir(bench, seed)
        os.makedirs(d, exist_ok=True)
        state[gpu] = tag
        print("[%5.1f min] gpu%d START %s" % ((time.time()-t0)/60, gpu, tag))
        cmd = ["python", TRAIN_PY, "--dataset", bench, "--gpu", "0", "--gpu_index", "0",
               "--model_name", ARM, "--batch_size", BS, "--seed", str(seed),
               "--data_dir", DATA_DIR, "--save_path", d,
               "--log_dir", "/tmp/logs/%s_%d" % (bench, seed)]
        p = subprocess.Popen(cmd, cwd=os.path.dirname(TRAIN_PY),
                             stdout=open(d + "/train.log", "w"),
                             stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        p.wait()
        ok = p.returncode == 0 and job_done(bench, seed)
        print("[%5.1f min] gpu%d %s -> %s"
              % ((time.time()-t0)/60, gpu, tag, "ok" if ok else "FAILED rc=%s" % p.returncode))
        if ok:
            subprocess.run("rclone copy %s %s/results/bisel8_cls/%s/seed_%d "
                           "--drive-chunk-size 64M" % (d, REMOTE, bench, seed),
                           shell=True, check=False)
        else:
            # 40 lines, not 15: the previous failure scrolled off and the run had
            # to be repeated blind.
            print("----- last 40 log lines: %s -----" % tag)
            print("".join(open(d + "/train.log").readlines()[-40:]))
    state[gpu] = "done"

threads = [threading.Thread(target=run_queue, args=(g, j), daemon=True)
           for g, j in PLAN.items()]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    time.sleep(300)
    print("   [%5.1f min] %s" % ((time.time()-t0)/60,
          " | ".join("gpu%d: %s" % (g, state[g]) for g in sorted(state))))
for t in threads:
    t.join()
print("")
print("done in %.1f min" % ((time.time() - t0) / 60))


In [ ]:

# -- Cell 7 -- score, with paired-friendly bootstrap CIs.
#
# Both benchmarks take the 5-fold branch, so each fold predicts the full test set
# and the folds are ensembled by MEAN LOGIT, thresholded at 0 -- the aggregation
# reverse-engineered from their shipped CellPPD predictions during the
# reproduction.
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score

subprocess.run("rclone copy %s/results/bisel8_cls %s --transfers 8 -P" % (REMOTE, OUT),
               shell=True, check=False)
rng = np.random.default_rng(0)
rows = []
for f in sorted(glob.glob(OUT + "/*/seed_*/*_results.csv")):
    seed = int(os.path.basename(os.path.dirname(f)).split("_")[1])
    bench = os.path.basename(os.path.dirname(os.path.dirname(f)))
    d = pd.read_csv(f)
    if "fold" in d.columns and d.fold.nunique() > 1:
        d["i"] = d.groupby("fold").cumcount(); g = d.groupby("i")
        y, p = g.true_label.first().values, g.predicted_label.mean().values
    else:
        y, p = d.true_label.values, d.predicted_label.values
    bs = []
    for _ in range(2000):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        bs.append(matthews_corrcoef(y[i], (p[i] > 0).astype(int)))
    rows.append(dict(bench=bench, seed=seed, n=len(y),
                     mcc=round(matthews_corrcoef(y, (p > 0).astype(int)), 4),
                     lo=round(float(np.percentile(bs, 2.5)), 4),
                     hi=round(float(np.percentile(bs, 97.5)), 4),
                     auc=round(roc_auc_score(y, p), 4)))
res = pd.DataFrame(rows).sort_values(["bench", "seed"])
print(res.to_string(index=False))

for b in res.bench.unique():
    s = res[res.bench == b]
    print("\n%s: bisel8 84.8M  MCC %.4f +- %.4f over %d seeds"
          % (b, s.mcc.mean(), s.mcc.std(), len(s)))

print("\n--- THPep reference, same 5-fold protocol ---")
for k, v in [("prefix16 (0-15) 168.8M", 0.8531), ("bisel8 seed101, earlier run", 0.8476),
             ("warmstart32M 31.7M", 0.8431), ("trunc24 (0-23)", 0.8218),
             ("trunc8 (0-7) 84.8M", 0.8037), ("full337M teacher", 0.7764),
             ("their published mlm-large", 0.7557), ("bag-of-tokens control", 0.6854)]:
    print("   %-30s %.4f" % (k, v))

print("\n--- CellPPD reference ---")
for k, v in [("our reproduction, mlm-large 337M", 0.8883), ("their published mlm-large", 0.8750),
             ("warmstart32M 31.7M", 0.8473), ("BAG-OF-TOKENS CONTROL", 0.8270),
             ("treatment (cached KD) 32M", 0.8273), ("kd-live 32M", 0.8254)]:
    print("   %-34s %.4f" % (k, v))
print("   ^ the control ties the 337M encoder, so CellPPD cannot separate backbones.")
print("     Report bisel8's number for completeness; do not read it as evidence.")

res.to_csv(OUT + "/bisel8_cls_metrics.csv", index=False)
subprocess.run("rclone copy %s %s/results/bisel8_cls --drive-chunk-size 64M -P"
               % (OUT, REMOTE), shell=True, check=True)
print("\nuploaded -> %s/results/bisel8_cls" % REMOTE)
